In [1]:
!python3 -m pip install joblib
!python3 -m pip install scikit-learn joblib pandas


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Load dataset
df = pd.read_csv("data/weatherAUS.csv")

df["RainTomorrow"] = df["RainTomorrow"].map({"Yes": 1, "No": 0})

# Drop rows with missing target
df = df.dropna(subset=["RainTomorrow"])

# Select full feature set
X = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

# All numeric features
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# All categorical features
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

# Full model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

# Train model
model.fit(X, y)

# Save compressed model
joblib.dump(model, "model/aussie_rain.joblib", compress=("lzma", 6))

print("Model saved successfully!")


Model saved successfully!


In [16]:
import numpy as np

feature_names = preprocessor.get_feature_names_out()

importances = rf.feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

fi.head(20)


,feature,importance
9,num__Humidity3pm,0.095050
8,num__Humidity9am,0.041254
11,num__Pressure3pm,0.040799
10,num__Pressure9am,0.039365
4,num__Sunshine,0.039230
5,num__WindGustSpeed,0.037964
15,num__Temp3pm,0.037590
2,num__Rainfall,0.035625
1,num__MaxTemp,0.034504
0,num__MinTemp,0.034024


In [2]:
import joblib
m = joblib.load("model/aussie_rain.joblib")
joblib.dump(m, "model/aussie_rain.joblib", compress=("lzma", 9))


['model/aussie_rain.joblib']

In [3]:
ls -lh model/


total 196672
-rw-r--r--@ 1 oleksandraandriyishyn  staff    93M  6 гру 22:09 aussie_rain.joblib


In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/weatherAUS.csv")
df["RainTomorrow"] = df["RainTomorrow"].map({"Yes": 1, "No": 0})
df = df.dropna(subset=["RainTomorrow"])

X = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

num_cols = X.select_dtypes(include=["float64", "int64"]).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

num_medians = X[num_cols].median()
cat_modes = X[cat_cols].mode().iloc[0]

X[num_cols] = X[num_cols].fillna(num_medians)
X[cat_cols] = X[cat_cols].fillna(cat_modes)

X = pd.get_dummies(X, drop_first=True)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

preprocessor_data = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "num_medians": num_medians,
    "cat_modes": cat_modes,
    "dummy_columns": X.columns.tolist()
}

joblib.dump(preprocessor_data, "model/preprocessor.joblib", compress=("zlib", 3))


['model/aussie_rain.joblib']

In [5]:
ls -lh model/


total 361512
-rw-r--r--@ 1 oleksandraandriyishyn  staff   172M  6 гру 22:15 aussie_rain.joblib
-rw-r--r--  1 oleksandraandriyishyn  staff    11K  6 гру 22:14 preprocessor.joblib


In [10]:
aussie_rain = joblib.load('model/aussie_rain.joblib')


In [11]:
joblib.dump(aussie_rain, "model/aussie_rain.joblib", compress=("lzma", 9))

['model/aussie_rain.joblib']

In [12]:
ls -lh model/


total 164576
-rw-r--r--@ 1 oleksandraandriyishyn  staff    76M  6 гру 22:27 aussie_rain.joblib
-rw-r--r--  1 oleksandraandriyishyn  staff    11K  6 гру 22:14 preprocessor.joblib
